# 02b Initialization from your own cell table (without QuPath)

Use this notebook instead of `02a_initialization_qupath.ipynb` if your segmentation, measurements and marker calls come from another tool. QuantCell needs one table with **one row per cell** containing:

| Column | Content |
|---|---|
| `x`, `y` | Cell centroid (any unit, used only to separate tissue sections) |
| `sample` | Sample or image the cell belongs to (one value is fine) |
| `<Marker>: <Compartment>: <Statistic>` | Quantitative features. Compartment is `Nucleus`, `Cytoplasm`, `Membrane` or `Cell`; statistic is `Mean`, `Median`, `Min`, `Max` or `Std.Dev.` (for example `CD3: Cell: Mean`). Any subset works; marker names must be letters, digits or `_`. |
| one call column per marker | Call for every marker in the marker combinations: positive (`True`, `1`, `+`), negative (`False`, `0`, `-`) or unknown (`?`, empty). A cell with an unknown call cannot match a combination that uses that marker. |

Other columns are ignored. To use differently named feature columns, set `project.feature_regex` in `03_model_selection.ipynb` and `04_annotation.ipynb` before `process_data()`.

To have a table to work with, this notebook first builds one from the sample QuPath exports, then runs the steps you would run on your own table. At the end it checks that the result matches `02a_initialization_qupath.ipynb`.

In [1]:
import os, json, glob, warnings
import pandas as pd, numpy as np

os.chdir('../Quantcell')
from codex_project import codex_project
warnings.filterwarnings('ignore')

In [2]:
annotations_path = '../marker_combos_062525_updated_verified.json'
base_dir = '../Output'
project_name = 'QuantCellPaperO3_custom'      # output folder for this route
project_path = f'{base_dir}/{project_name}'
os.makedirs(project_path, exist_ok=True)

marker_combos = json.load(open(annotations_path))
markers = sorted({m[:-1] for combo in marker_combos.values() for m in combo})
print(len(markers), 'markers:', ', '.join(markers))

25 markers: B220, CD105, CD115, CD117, CD11b, CD11c, CD127, CD19, CD3, CD31, CD34, CD4, CD41, CD45, CD71, CD8, Endomucin, F4_80, Fcgr, Lepr, Ly6g, MHCII, Sca1, TCR, Ter119


### Example input
Skip this cell with your own data and load your table instead, for example `table = pd.read_csv('my_cells.csv')`.

In [3]:
# Build an example table from the sample QuPath exports: features + one True/False call column per marker
files = {os.path.basename(f)[:-4]: f for f in glob.glob('../sample_data/QuantCellPaperO3/*.csv')}
first = pd.read_csv(files[markers[0]])
table = pd.DataFrame({'x': first['Centroid X µm'], 'y': first['Centroid Y µm'], 'sample': 'QuantCellPaperO3'})
for m in markers:
    d = pd.read_csv(files[m])
    feats = [c for c in d.columns if c.startswith(f'{m}: ')]
    table = pd.concat([table, d[feats]], axis=1)
    name = d['Name'].astype(str)      # QuPath class: the marker's class name if positive, 'Other' if negative, 'PathCellObject' if unclassified
    table[m] = np.where(name == 'Other', '-', np.where(name.str.upper() == 'PATHCELLOBJECT', '?', '+'))
table.iloc[:3, :8]

,x,y,sample,B220,CD105: Nucleus: Mean,CD105: Nucleus: Median,CD105: Nucleus: Min,CD105: Nucleus: Max
0,2592.3,2536.0,QuantCellPaperO3,-,4.0909,0.0,0,55
1,2563.6,2538.0,QuantCellPaperO3,-,14.1264,3.0,0,113
2,2603.7,2538.8,QuantCellPaperO3,-,0.0000,0.0,0,0


### 1. Encode the marker calls
QuantCell stores each cell's calls as one string, for example `:CD3+:CD4-:...`. This converts your call columns to that format.

In [4]:
def to_call(v):
    if pd.isna(v): return '?'
    s = str(v).strip().lower()
    if s in ('true', '1', '1.0', '+', 'pos', 'positive'): return '+'
    if s in ('false', '0', '0.0', '-', 'neg', 'negative'): return '-'
    if s in ('?', '', 'unknown', 'nan'): return '?'
    raise ValueError(f'unrecognised call value: {v!r}')

missing = [m for m in markers if m not in table.columns]
assert not missing, f'no call column for: {missing}'
calls = {m: table[m].map(to_call) for m in markers}
table['marker_string'] = [''.join(f':{m}{calls[m].iat[i]}' for m in markers) for i in range(len(table))]
table['marker_string'].iat[0]

':B220-:CD105-:CD115-:CD117-:CD11b-:CD11c-:CD127-:CD19-:CD3+:CD31-:CD34-:CD4-:CD41-:CD45-:CD71-:CD8-:Endomucin-:F4_80-:Fcgr-:Lepr-:Ly6g-:MHCII-:Sca1+:TCR+:Ter119-'

### 2. Rule-based annotation
Cells matching exactly one marker combination get that cell type. Cells matching none, or more than one, are left unannotated (`Other`).

In [5]:
codex = codex_project()
codex.read_annotation_strategy(annotations_path)
codex.codex = table.drop(columns=markers)              # keep features, coordinates, sample and marker_string
codex.codex['original_folder'] = project_name
codex.codex['annotation_strategy'] = codex.annotation_strategy
codex._sample_named = True
codex.annotate()
codex.codex['cell_type'].value_counts()

Other                       18568
Erythroblasts                7479
Megakaryocytes               5725
B Cells                      3689
Erythrocytes                 1307
Neutrophils                  1255
cDC                           873
CD8 T Cells                   591
CD4 T Cells                   557
Lepr+ Perivascular Cells      515
Arterial ECs                  446
Capillary ECs                 368
Other BM ECs                  218
Monocytes                     163
Double Negative T Cells       160
MEP                           148
MkP                           139
CMP                            71
CLP                            59
KLS                            54
CFU-E                          27
GMP                            22
Name: cell_type, dtype: int64

### 3. Tissue sections
`section_samples()` separates spatially distinct tissue pieces with DBSCAN (`eps` in the units of `x`, `y`). Use `override_sectioning()` instead to treat each sample as one section.

In [6]:
codex.section_samples(eps=500)
codex.verify_sectioning()

~~~Verifying sectioning~~~
Passed: All cells assigned in QuantCellPaperO3_custom


True

### 4. Save
The file name `codex_rule_based_<project>.csv` is what `03_model_selection.ipynb` and `04_annotation.ipynb` load (set `project_name` there to this route's name).

In [7]:
print('table valid:', codex.verify_codex(verbose=False))
codex.save_csv(f'{project_path}/codex_rule_based_{project_name}.csv')

table valid: True


### Check: same result as the QuPath route
Only possible here because the example table came from the sample QuPath exports.

In [8]:
qupath_route = pd.read_csv('../Output/QuantCellPaperO3/codex_rule_based_QuantCellPaperO3.csv', usecols=['cell_type'])
print('cells:', len(qupath_route), 'vs', len(codex.codex))
print('cell types identical to 02a:', bool((qupath_route['cell_type'].values == codex.codex['cell_type'].values).all()))

cells: 42434 vs 42434
cell types identical to 02a: True
